In [ ]:
from pathlib import Path
import os
import json

os.environ.setdefault(
    "MPLCONFIGDIR",
    str((Path.cwd() / ".." / ".tmp" / "matplotlib").resolve()),
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

try:
    from IPython.display import display
except ImportError:
    display = print

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "font.size": 10,
})

def find_project_dir():
    here = Path.cwd().resolve()
    candidates = [
        here,
        here / "stanage_eda",
        here / "HPC carbon release" / "stanage_eda",
        here.parent / "stanage_eda",
        here.parent / "HPC carbon release" / "stanage_eda",
        Path.home() / "Desktop" / "HPC carbon release" / "stanage_eda",
    ]

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "results" / "full_2025" / "analysis_summary.json").exists():
            return candidate

    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "Could not find Stanage EDA results/full_2025/analysis_summary.json. "
        f"Checked:\n{checked}"
    )


PROJECT_DIR = find_project_dir()

RESULTS = PROJECT_DIR / "results" / "full_2025"
FIGURES = PROJECT_DIR / "outputs" / "notebook_figures"
FIGURES.mkdir(parents=True, exist_ok=True)

def load_csv(name):
    return pd.read_csv(RESULTS / name)

with (RESULTS / "analysis_summary.json").open(encoding="utf-8") as handle:
    summary = json.load(handle)

monthly = load_csv("monthly_submit_summary.csv")
states = load_csv("state_summary.csv")
distributions = load_csv("distribution_summary.csv")
hours = load_csv("submission_hour_utc.csv")
weekdays = load_csv("submission_weekday_utc.csv")
partitions = load_csv("partition_summary.csv")
user_buckets = load_csv("user_workload_buckets.csv")
node_groups = load_csv("node_group_summary.csv")
fields = load_csv("field_availability.csv")
power_inputs = load_csv("energy_power_assumptions.csv")
simulator_sample = load_csv("simulator_workload_sample.csv")

def save_figure(name):
    plt.tight_layout()
    plt.savefig(FIGURES / name, bbox_inches="tight")
    plt.show()

In [ ]:
headline = pd.DataFrame({
    "Measure": [
        "Raw records", "Unique 2025 jobs", "Duplicate records removed",
        "Anonymous users", "Completed jobs", "Completion rate",
        "Rows with non-zero job energy", "Rows with non-zero CPU utilisation",
    ],
    "Value": [
        f"{summary['counts']['raw_rows']:,}",
        f"{summary['counts']['cohort_jobs_submitted_in_2025']:,}",
        f"{summary['counts']['duplicate_rows']:,}",
        f"{summary['counts']['unique_users']:,}",
        f"{summary['outcomes']['completed_jobs']:,}",
        f"{summary['outcomes']['completion_rate_pct']:.1f}%",
        f"{summary['energy_evidence']['nonzero_consumed_energy_raw_rows']:,}",
        f"{summary['energy_evidence']['nonzero_total_cpu_rows']:,}",
    ],
})
display(headline)

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(monthly))
ax1.bar(x, monthly["jobs"] / 1000, color="#2F6B9A", width=0.68, label="Submitted jobs")
ax1.set_ylabel("Submitted jobs (thousands)")
ax1.set_xticks(x, monthly["submit_month"].str[-2:])
ax1.set_xlabel("Month in 2025 (UTC submit time)")

ax2 = ax1.twinx()
ax2.plot(x, monthly["completion_rate_pct"], color="#188977", marker="o", linewidth=2.3,
         label="Completion rate")
ax2.set_ylabel("Completion rate (%)")
ax2.set_ylim(0, 100)

lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc="upper right")
ax1.set_title("Monthly workload and completion rate")
save_figure("01_monthly_workload.png")

display(monthly[["submit_month", "jobs", "completion_rate_pct", "gpu_jobs",
                 "median_runtime_min", "p95_runtime_min"]].round(2))

In [ ]:
plot_states = states.sort_values("jobs", ascending=True)
colors = ["#188977" if c == "completed" else "#D49A35" if c == "failure" else "#8A96A3"
          for c in plot_states["category"]]
plt.figure(figsize=(10, 5.2))
plt.barh(plot_states["state"], plot_states["jobs"] / 1000, color=colors)
plt.xlabel("Jobs (thousands)")
plt.title("Final job states")
save_figure("02_job_states.png")
display(states.round(3))

In [ ]:
selected_metrics = [
    "submit_wait_sec", "eligible_wait_sec", "runtime_sec", "requested_cpus",
    "allocated_nodes", "requested_gpus", "timelimit_utilisation",
]
dist_view = distributions.loc[distributions["metric"].isin(selected_metrics),
                              ["metric", "unit", "valid_count", "median", "p95", "p99", "maximum"]]
display(dist_view.round(3))

wait_runtime = distributions.set_index("metric").loc[
    ["submit_wait_sec", "eligible_wait_sec", "runtime_sec"], ["median", "p95"]
]
wait_runtime.index = ["Submit to start (A)", "Eligible to start (B)", "Runtime (C)"]
ax = wait_runtime.plot(kind="bar", color=["#2F6B9A", "#D49A35"], figsize=(10, 5.4))
ax.set_yscale("log")
ax.set_ylabel("Minutes (log scale)")
ax.set_xlabel("")
ax.set_title("Median and P95 expose a long-tailed workload")
ax.tick_params(axis="x", rotation=0)
save_figure("03_wait_runtime_distribution.png")

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekdays_plot = weekdays.set_index("weekday").reindex(weekday_order).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].bar(hours["utc_hour"], hours["share_pct"], color="#2F6B9A")
axes[0].set_title("Submissions by hour")
axes[0].set_xlabel("Hour (UTC)")
axes[0].set_ylabel("Share of jobs (%)")
axes[0].set_xticks(range(0, 24, 3))

axes[1].bar(weekdays_plot["weekday"].str[:3], weekdays_plot["share_pct"], color="#188977")
axes[1].set_title("Submissions by weekday")
axes[1].set_xlabel("Weekday (UTC)")
axes[1].set_ylabel("Share of jobs (%)")
save_figure("04_arrival_patterns.png")

In [ ]:
top_partitions = partitions.nlargest(8, "jobs").copy()
plt.figure(figsize=(10.5, 5.2))
plt.barh(top_partitions["partition"][::-1], top_partitions["jobs"][::-1] / 1000,
         color="#2F6B9A")
plt.xlabel("Jobs (thousands)")
plt.title("Largest partitions by submitted jobs")
save_figure("05_partition_volume.png")

display(top_partitions[["partition", "jobs", "share_pct", "completion_rate_pct",
                        "median_submit_wait_min", "p95_submit_wait_min", "gpu_jobs"]].round(2))

In [ ]:
display(user_buckets.round(4))
heavy = user_buckets.loc[user_buckets["jobs_per_user_bucket"] == "1001+"].iloc[0]
print(f"{int(heavy['users'])} users with more than 1,000 jobs generated "
      f"{heavy['job_share_pct']:.2f}% of all jobs.")

In [ ]:
display(node_groups)
print(f"Observed nodes across all expanded NodeList expressions: "
      f"{node_groups['observed_nodes'].sum():,.0f}")

In [ ]:
energy_fields = fields.loc[
    fields["field"].isin(["ConsumedEnergy", "ConsumedEnergyRaw", "TotalCPU", "UserCPU",
                          "SystemCPU", "TRESUsageInAve", "TRESUsageInMax"]),
    ["field", "nonempty_rows", "nonempty_rate_pct", "nonzero_rows"]
]
display(energy_fields.fillna("not applicable"))
display(power_inputs)

In [ ]:
print(f"Identity-free calibration sample: {len(simulator_sample):,} jobs")
display(simulator_sample.head())
display(simulator_sample.dtypes.rename("dtype").to_frame())